# Benchmark Result Statistics

This notebook calculates summary statistics for RV benchmark sessions.

For both **baseline** and **abnormal** datasets, it computes:
- `elapsedMs` stats (count, mean, stddev, median, min/max, p95, p99)
- `rowsPerSec` stats (count, mean, stddev, median, min/max, p95, p99)
- Overhead percentage from average elapsed time.

In [1]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path
from statistics import mean, median, stdev


def percentile(values: list[float], p: float) -> float:
    if not values:
        return float('nan')
    if len(values) == 1:
        return values[0]
    s = sorted(values)
    rank = (p / 100.0) * (len(s) - 1)
    lower = int(math.floor(rank))
    upper = int(math.ceil(rank))
    if lower == upper:
        return s[lower]
    w = rank - lower
    return s[lower] + (s[upper] - s[lower]) * w


def describe(values: list[float]) -> dict[str, float]:
    if not values:
        return {
            'count': 0,
            'mean': float('nan'),
            'stddev': float('nan'),
            'median': float('nan'),
            'min': float('nan'),
            'max': float('nan'),
            'p95': float('nan'),
            'p99': float('nan'),
        }
    return {
        'count': len(values),
        'mean': mean(values),
        'stddev': stdev(values) if len(values) > 1 else 0.0,
        'median': median(values),
        'min': min(values),
        'max': max(values),
        'p95': percentile(values, 95.0),
        'p99': percentile(values, 99.0),
    }


def load_runs(csv_path: Path) -> dict[str, list[float]]:
    on_ms, off_ms, on_rps, off_rps = [], [], [], []
    with csv_path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            on_ms.append(float(row['onElapsedMs']))
            off_ms.append(float(row['offElapsedMs']))
            on_rps.append(float(row['onRowsPerSec']))
            off_rps.append(float(row['offRowsPerSec']))
    return {
        'onElapsedMs': on_ms,
        'offElapsedMs': off_ms,
        'onRowsPerSec': on_rps,
        'offRowsPerSec': off_rps,
    }


def analyze_mode(mode_name: str, csv_path: Path) -> dict:
    runs = load_runs(csv_path)
    on_ms_stats = describe(runs['onElapsedMs'])
    off_ms_stats = describe(runs['offElapsedMs'])
    on_rps_stats = describe(runs['onRowsPerSec'])
    off_rps_stats = describe(runs['offRowsPerSec'])

    overhead_pct = ((on_ms_stats['mean'] - off_ms_stats['mean']) / off_ms_stats['mean']) * 100.0

    return {
        'mode': mode_name,
        'sourceCsv': str(csv_path),
        'rvOn': {'elapsedMs': on_ms_stats, 'rowsPerSec': on_rps_stats},
        'rvOff': {'elapsedMs': off_ms_stats, 'rowsPerSec': off_rps_stats},
        'overheadPct': overhead_pct,
    }


def latest_session_dir(runs_dir: Path) -> Path:
    sessions = sorted([p for p in runs_dir.glob('session-*') if p.is_dir()])
    if not sessions:
        raise FileNotFoundError(f'No session-* folders found in: {runs_dir}')
    return sessions[-1]


def format_block(title: str, stats: dict[str, float], unit: str = '') -> str:
    return (
        f"{title}\n"
        f"  mean={stats['mean']:.4f}{unit} stddev={stats['stddev']:.4f}{unit} median={stats['median']:.4f}{unit}\n"
        f"  min={stats['min']:.4f}{unit} max={stats['max']:.4f}{unit} p95={stats['p95']:.4f}{unit} p99={stats['p99']:.4f}{unit}"
    )

In [2]:
# Configuration
# Optional override: set to a specific folder name such as 'session-20260506-100232'
FORCE_SESSION = None


def resolve_runs_dir(start: Path) -> Path:
    """Find runs directory even if notebook is launched elsewhere."""
    candidates = [
        start / 'runs',
        start / 'src' / 'resultCollection' / 'runs',
    ]
    for parent in [start] + list(start.parents):
        candidates.append(parent / 'runs')
        candidates.append(parent / 'src' / 'resultCollection' / 'runs')

    for c in candidates:
        if c.exists() and c.is_dir() and any(c.glob('session-*')):
            return c
    raise FileNotFoundError(
        f'Could not locate a runs directory from start path: {start}. '
        'Expected runs/session-* or src/resultCollection/runs/session-*.'
    )


start_dir = Path.cwd()
runs_dir = resolve_runs_dir(start_dir)

if FORCE_SESSION:
    session_dir = runs_dir / FORCE_SESSION
else:
    session_dir = latest_session_dir(runs_dir)

baseline_csv = session_dir / 'baseline' / 'benchmark-runs.csv'
abnormal_csv = session_dir / 'abnormal' / 'benchmark-runs.csv'

if not baseline_csv.exists() or not abnormal_csv.exists():
    raise FileNotFoundError('Missing benchmark-runs.csv in baseline/abnormal folders for selected session.')

baseline = analyze_mode('baseline', baseline_csv)
abnormal = analyze_mode('abnormal', abnormal_csv)

report = {
    'sessionDir': str(session_dir),
    'baseline': baseline,
    'abnormal': abnormal,
}

print(f'Runs directory: {runs_dir}')
print(f'Session: {session_dir}')
print('Computed statistics for baseline and abnormal.')

Runs directory: c:\Users\brama\Documents\RD2\src\resultCollection\runs
Session: c:\Users\brama\Documents\RD2\src\resultCollection\runs\session-20260506-100232
Computed statistics for baseline and abnormal.


In [3]:
# Show results as tables
from IPython.display import display


def section_to_rows(section: dict, metric_key: str) -> list[dict]:
    rows = []
    for mode_label, mode_key in [('rvOn', 'rvOn'), ('rvOff', 'rvOff')]:
        s = section[mode_key][metric_key]
        rows.append({
            'mode': mode_label,
            'count': s['count'],
            'mean': s['mean'],
            'stddev': s['stddev'],
            'median': s['median'],
            'min': s['min'],
            'max': s['max'],
            'p95': s['p95'],
            'p99': s['p99'],
        })
    return rows


try:
    import pandas as pd

    overhead_df = pd.DataFrame([
        {
            'dataset': 'baseline',
            'source': baseline['sourceCsv'],
            'overheadPct': baseline['overheadPct'],
        },
        {
            'dataset': 'abnormal',
            'source': abnormal['sourceCsv'],
            'overheadPct': abnormal['overheadPct'],
        },
    ])

    print('Overhead Summary')
    display(overhead_df)

    print('Baseline - elapsedMs')
    display(pd.DataFrame(section_to_rows(baseline, 'elapsedMs')))

    print('Baseline - rowsPerSec')
    display(pd.DataFrame(section_to_rows(baseline, 'rowsPerSec')))

    print('Abnormal - elapsedMs')
    display(pd.DataFrame(section_to_rows(abnormal, 'elapsedMs')))

    print('Abnormal - rowsPerSec')
    display(pd.DataFrame(section_to_rows(abnormal, 'rowsPerSec')))

except ImportError:
    # Fallback if pandas is unavailable
    print('pandas not installed; printing JSON-formatted tables.')
    print(json.dumps({'baseline': baseline, 'abnormal': abnormal}, indent=2))

Overhead Summary


,dataset,source,overheadPct
0,baseline,c:\Users\brama\Documents\RD2\src\resultCollect...,8.341323
1,abnormal,c:\Users\brama\Documents\RD2\src\resultCollect...,3951.943128


Baseline - elapsedMs


,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,10,113.0,7.180220,110.5,109.0,133.0,124.0,131.20
1,rvOff,10,104.3,8.393781,102.0,100.0,128.0,117.2,125.84


Baseline - rowsPerSec


,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,10,607526.573810,33682.841528,619315.847666,514533.834586,627825.688073,627825.688073,627825.688073
1,rvOff,10,659328.285304,44358.480813,670911.764706,534632.812500,684330.000000,681281.004950,683720.200990


Abnormal - elapsedMs


,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,10,4274.8,137.395455,4266.5,4086.0,4539.0,4497.6,4530.72
1,rvOff,10,105.5,4.089281,104.0,100.0,111.0,111.0,111.00


Abnormal - rowsPerSec


,mode,count,mean,stddev,median,min,max,p95,p99
0,rvOn,10,16023.113920,506.588715,16039.611143,15076.668870,16748.164464,16628.361842,16724.203940
1,rvOff,10,649524.922161,24972.423565,658009.615385,616513.513514,684330.000000,678291.794118,683122.358824


In [4]:
# Export JSON and TXT side-by-side with session outputs
json_out = session_dir / 'stats-summary.json'
txt_out = session_dir / 'stats-summary.txt'

json_out.write_text(json.dumps(report, indent=2), encoding='utf-8')

lines = ['Benchmark Statistics Summary', f'session={session_dir}', '']
for label, section in [('BASELINE', baseline), ('ABNORMAL', abnormal)]:
    lines.append(f'[{label}]')
    lines.append(f"source={section['sourceCsv']}")
    lines.append(f"overheadPct={section['overheadPct']:.4f}")
    lines.append(format_block('rvOn.elapsedMs', section['rvOn']['elapsedMs'], 'ms'))
    lines.append(format_block('rvOff.elapsedMs', section['rvOff']['elapsedMs'], 'ms'))
    lines.append(format_block('rvOn.rowsPerSec', section['rvOn']['rowsPerSec']))
    lines.append(format_block('rvOff.rowsPerSec', section['rvOff']['rowsPerSec']))
    lines.append('')

txt_out.write_text('\n'.join(lines), encoding='utf-8')

print(f'Wrote: {json_out}')
print(f'Wrote: {txt_out}')

Wrote: c:\Users\brama\Documents\RD2\src\resultCollection\runs\session-20260506-100232\stats-summary.json
Wrote: c:\Users\brama\Documents\RD2\src\resultCollection\runs\session-20260506-100232\stats-summary.txt
